In [7]:
import requests
import json
import pandas as pd
import datetime


# -------------------------------------------
# CONFIGURATION
# -------------------------------------------
# API_KEY = "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"
API_KEY = "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"
MODEL = "openai/gpt-4.1"
ENDPOINT = "https://openrouter.ai/api/v1/chat/completions"


# -------------------------------------------
# PROMPT TEMPLATE (index-aligned, no markdown)
# -------------------------------------------
PROMPT_TEMPLATE = """
SYSTEM:
You are a company name standardization system.

TASK:
For each input company name, produce exactly one standardized company name.

STRICT RULES (NO EXCEPTIONS):
1. Output MUST be valid JSON only.
2. Do NOT use markdown, code blocks, or ```json.
3. Output length MUST equal input length.
4. Each input at index i MUST produce exactly one output at index i.
5. Do NOT skip, merge, reorder, or remove items.
6. It is allowed and expected to repeat standardized names.
7. Before responding, verify output count equals input count.

INPUT COMPANY NAMES (INDEXED):
{indexed_inputs}

REQUIRED OUTPUT FORMAT (JSON ONLY):

{{
  "standardized_names": [
    "<output for index 0>",
    "<output for index 1>",
    "<output for index 2>"
  ]
}}
"""

In [8]:

def format_indexed_inputs(name_list):
    return json.dumps(
        [f"{i}: {name}" for i, name in enumerate(name_list)],
        ensure_ascii=False
    )


def clean_llm_output(text):
    text = text.strip()
    if text.startswith("```"):
        text = text.split("```", 1)[-1]
        if text.lstrip().startswith("json"):
            text = text.lstrip()[4:]
        text = text.strip("`\n")
    return text.strip()


def standardize_names(name_list, temperature=0.3, top_p=1.0, max_retries=5):
    n = len(name_list)

    prompt = PROMPT_TEMPLATE.format(
        indexed_inputs=format_indexed_inputs(name_list)
    )

    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {API_KEY}",
        "HTTP-Referer": "http://localhost",
        "X-Title": "Name Standardizer",
    }

    body = {
        "model": MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": temperature,
        "top_p": top_p
    }

    last_error = None

    for _ in range(max_retries):
        response = requests.post(
            ENDPOINT, headers=headers, json=body, timeout=60
        )
        response.raise_for_status()

        raw = response.json()["choices"][0]["message"]["content"]
        cleaned = clean_llm_output(raw)

        try:
            data = json.loads(cleaned)
            names = data["standardized_names"]

            if not isinstance(names, list):
                raise ValueError("standardized_names is not a list")

            if len(names) != n:
                raise ValueError(f"Expected {n}, got {len(names)}")

            return names

        except Exception as e:
            last_error = e

    raise RuntimeError(f"Failed after retries: {last_error}")

In [9]:
def evaluate_model(
    data,
    batch_size=20,
    runs=10,
    temperature=0.3,
    top_p=1.0,
    std_name_list=standardize_names,
):
    for k in range(1, runs + 1):
        print(f"Run {k}: {datetime.datetime.now()}")

        predictions = []
        not_canonical = []
        for i in range(0, len(data), batch_size):
            batch = data.iloc[i:i+batch_size]["variant"].tolist()
            preds = standardize_names(
                batch,
                temperature=temperature,
                top_p=top_p
            )
            predictions.extend(preds)

            for name in preds:
                if name not in std_name_list:
                    not_canonical.append(name)

        data_run = data.copy()
        data_run["prediction"] = predictions
        data_run["exact_match"] = (
            data_run["prediction"] == data_run["canonical"]
        ).astype(int)

        summary = (
            data_run
            .groupby("type")["exact_match"]
            .sum()
            .to_frame(name="correct")
        )

        summary.to_csv(
            f"predictions_run_{k}_temp_{temperature}_p_{top_p}.csv"
        )

        print(f"Run {k} not_canonical: {len(not_canonical)}")

In [10]:
# data = pd.read_csv("cleaned_generated_variants.csv")
# data = data.sample(frac=1, random_state=123).reset_index(drop=True)
# print(data.iloc[:3000, :].groupby("type").size())

# data.iloc[:3000, :].to_csv("selected_data.csv", index=False)

In [11]:
data = pd.read_csv("selected_data2.csv")
std_names = data["canonical"].unique().tolist()
print(data.groupby("type").size())
print(data.shape[0])

type
abbreviation    124
brand_parent    196
legal_suffix     57
partial         303
punctuation     232
typography      588
dtype: int64
1500


In [12]:
# evaluate_model(data, batch_size=50, temperature=0, top_p=0.1)
# evaluate_model(data, batch_size=200, temperature=0, top_p=0.3)

# evaluate_model(data, batch_size=200, temperature=0.3, top_p=0.1)
# evaluate_model(data, batch_size=200, temperature=0.3, top_p=0.3)

evaluate_model(data, batch_size=200, temperature=0, top_p=0.3, std_name_list=std_names)

Run 1: 2026-04-05 14:00:39.898705
Run 1 not_canonical: 753
Run 2: 2026-04-05 14:01:57.904006
Run 2 not_canonical: 756
Run 3: 2026-04-05 14:03:02.109849
Run 3 not_canonical: 751
Run 4: 2026-04-05 14:04:02.430879
Run 4 not_canonical: 754
Run 5: 2026-04-05 14:05:12.306602
Run 5 not_canonical: 753
Run 6: 2026-04-05 14:06:17.936806
Run 6 not_canonical: 752
Run 7: 2026-04-05 14:07:19.225138
Run 7 not_canonical: 752
Run 8: 2026-04-05 14:08:20.097998
Run 8 not_canonical: 760
Run 9: 2026-04-05 14:09:21.411842
Run 9 not_canonical: 762
Run 10: 2026-04-05 14:10:29.343357
Run 10 not_canonical: 750


In [ ]:
import os

data_dir = os.path.join(os.getcwd(), "results", "base model results")
sample_count = data.groupby("type").size()

data1 = pd.read_csv(os.path.join(data_dir, "temp_0.3_p_0.1.csv"))/sample_count
data2 = pd.read_csv(os.path.join(data_dir, "temp_0.3_p_0.3.csv"))/sample_count
data3 = pd.read_csv(os.path.join(data_dir, "temp_0_p_0.1.csv"))/sample_count
data4 = pd.read_csv(os.path.join(data_dir, "temp_0_p_0.3.csv"))/sample_count

In [ ]:
import numpy as np

import matplotlib.pyplot as plt

# Combine all data and calculate means and stds
datasets = [data1, data2, data3, data4]
dataset_labels = ['temp_0_p_0.1', 'temp_0_p_0.3', 'temp_0.3_p_0.1', 'temp_0.3_p_0.3']
columns = data1.columns

# Calculate means and stds for each dataset
means = [df.mean() for df in datasets]
stds = [df.std() for df in datasets]

# Create bar plot
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(columns))
width = 0.2

for i, (mean, std, label) in enumerate(zip(means, stds, dataset_labels)):
    mean_aligned = mean.reindex(columns)
    std_aligned = std.reindex(columns)

    ax.bar(
        x + i * width,
        mean_aligned.values,
        width,
        label=label,
        yerr=std_aligned.values,
        capsize=5
    )

ax.set_xlabel('Columns')
ax.set_ylabel('Mean Accuracy')
ax.set_title('Model Performance Comparison')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(columns, rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()